In [14]:
#!/usr/bin/env python3
"""
Availability timeline plots: Stormwatch vs Princeton (per station)

UPDATE (per user):
- Generate plots for ALL Princeton stations (all columns except "Date"),
  even if the station is missing in Stormwatch.
- If a station is missing in Stormwatch, the Stormwatch row is blank but still labeled.
- If a station is missing in Princeton (shouldn't happen in this loop), Princeton row is blank but labeled.

Inputs
- Stormwatch per-station hourly CSVs:
  /mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/compiled_rawstyle_hourly_mm/per_station_hourly/
  Columns used: time_local, rain_mm
  Missing hours appear as blank cells -> NaN (treat as not available)

- Princeton Excel:
  /mnt/12TB/Sujan/Spatial_correlation/Codes/dependent_files/princeton.csv.xlsx
  Columns: Date + station columns named by station number (e.g., 1720)
  Missing values are -99 (treat as not available)

Output
- One PNG per station:
  /mnt/12TB/Sujan/Spatial_correlation/Codes/Spatial_correlation/station_<ID>_availability.png

Timezone
- America/Chicago (DST-aware)
- Princeton "Date" is assumed Chicago-local, tz-naive -> localized to America/Chicago
  Handles DST safely:
    - nonexistent (spring forward): shift_forward
    - ambiguous (fall back): NaT
- Stormwatch time_local: if tz-naive -> localized using same DST rules; if tz-aware -> converted
"""

from __future__ import annotations
from datetime import datetime

import sys
from pathlib import Path
import re
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates


# ------------------ PATHS ------------------
STORMWATCH_DIR = Path(
    "/mnt/12TB/Sujan/Spatial_correlation/Codes/Compiled_rain/compiled_rawstyle_hourly_mm/per_station_hourly/"
)

PRINCETON_XLSX = Path(
    "/mnt/12TB/Sujan/Spatial_correlation/Codes/dependent_files/princeton.csv.xlsx"
)

OUT_DIR = Path("/mnt/12TB/Sujan/Spatial_correlation/Codes/Spatial_correlation/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LOCAL_TZ = "America/Chicago"
PRINCETON_MISSING = -99
HOUR = pd.Timedelta(hours=1)


# ------------------ HELPERS ------------------
def station_id_from_filename(fp: Path) -> str:
    m = re.search(r"(\d+)", fp.stem)
    return m.group(1) if m else fp.stem


def ensure_chicago_tz(dt: pd.Series) -> pd.Series:
    """
    Ensure datetime is tz-aware America/Chicago, DST-safe.
    - If tz-naive: localize with nonexistent='shift_forward' and ambiguous='NaT'
    - If tz-aware: convert to America/Chicago
    """
    if getattr(dt.dt, "tz", None) is None:
        return dt.dt.tz_localize(
            LOCAL_TZ,
            nonexistent="shift_forward",
            ambiguous="NaT",
        )
    return dt.dt.tz_convert(LOCAL_TZ)


def availability_segments(times: pd.DatetimeIndex) -> list[tuple[pd.Timestamp, pd.Timestamp]]:
    """
    Convert hourly timestamps into contiguous segments [start, end_exclusive).
    """
    if len(times) == 0:
        return []

    t = times.sort_values()
    diffs = t[1:] - t[:-1]
    breaks = diffs > HOUR

    segs: list[tuple[pd.Timestamp, pd.Timestamp]] = []
    start = t[0]
    prev = t[0]

    for i, is_break in enumerate(breaks, start=1):
        if is_break:
            segs.append((start, prev + HOUR))
            start = t[i]
        prev = t[i]

    segs.append((start, prev + HOUR))
    return segs


def segments_to_brokenbar(segs: list[tuple[pd.Timestamp, pd.Timestamp]]) -> list[tuple[float, float]]:
    out = []
    for a, b in segs:
        a_num = mdates.date2num(a.to_pydatetime())
        b_num = mdates.date2num(b.to_pydatetime())
        out.append((a_num, b_num - a_num))
    return out

def build_stormwatch_index() -> dict[str, Path]:
    """
    Map station_id -> stormwatch file path.
    """
    if not STORMWATCH_DIR.exists():
        return {}

    out: dict[str, Path] = {}
    for fp in STORMWATCH_DIR.glob("*.csv"):
        if not fp.is_file():
            continue
        sid = station_id_from_filename(fp)
        # If duplicates exist, keep the first one deterministically
        out.setdefault(sid, fp)
    return out


def load_stormwatch_availability(fp: Path) -> pd.DatetimeIndex:
    """
    Return timestamps (America/Chicago) where Stormwatch has data (rain_mm not NaN).
    Robust to mixed offsets in time_local strings.
    """
    df = pd.read_csv(fp)
    if "time_local" not in df.columns or "rain_mm" not in df.columns:
        return pd.DatetimeIndex([])

    # Parse time_local robustly:
    # - Force UTC parsing so mixed offsets become a single tz-aware dtype
    t_utc = pd.to_datetime(df["time_local"], errors="coerce", utc=True)
    v = pd.to_numeric(df["rain_mm"], errors="coerce")

    ok = ~t_utc.isna()
    t_utc = t_utc.loc[ok]
    v = v.loc[ok]

    # Convert to Chicago tz
    t_local = t_utc.dt.tz_convert(LOCAL_TZ)

    avail = t_local.loc[v.notna()]
    return pd.DatetimeIndex(avail.values)



def load_princeton_availability(pr: pd.DataFrame, sid: str) -> pd.DatetimeIndex:
    """
    Return timestamps (America/Chicago) where Princeton has data (value != -99 and not NaN).
    """
    if sid not in pr.columns:
        return pd.DatetimeIndex([])

    p_vals = pd.to_numeric(pr[sid], errors="coerce")
    mask = p_vals.notna() & (p_vals != PRINCETON_MISSING)
    avail = pr.loc[mask, "Date"]
    return pd.DatetimeIndex(avail.values)


# ------------------ MAIN ------------------
def main() -> None:

    sw_map = build_stormwatch_index()
    sw_stations = list(sw_map.keys())

    all_stations = sorted(
        set(pr_stations) | set(sw_stations),
        key=lambda x: (int(x) if str(x).isdigit() else str(x))
    )

    print(f"Stormwatch stations available: {len(sw_map)}")
    print(f"Total stations to plot (union): {len(all_stations)}")
    print(f"Output dir: {OUT_DIR}")

    # Fixed x-limits
    start_date = datetime(2012, 1, 1)
    end_date = datetime(2025, 12, 31)

    for sid in all_stations:  # ✅ IMPORTANT: use union
        pr_times = load_princeton_availability(pr, sid)
        pr_bars = segments_to_brokenbar(availability_segments(pr_times))

        sw_times = pd.DatetimeIndex([])
        if sid in sw_map:
            sw_times = load_stormwatch_availability(sw_map[sid])
        sw_bars = segments_to_brokenbar(availability_segments(sw_times))

        # Skip if truly nothing in either dataset
        if len(pr_times) == 0 and len(sw_times) == 0:
            continue

        fig, ax = plt.subplots(figsize=(14, 2.2))

        y_pr, y_sw, h = 0, 10, 7
        if pr_bars:
            ax.broken_barh(pr_bars, (y_pr, h))
        if sw_bars:
            ax.broken_barh(sw_bars, (y_sw, h))

        ax.set_yticks([y_pr + h / 2, y_sw + h / 2])
        ax.set_yticklabels(["Princeton", "Stormwatch"])

        ax.set_xlim(mdates.date2num(start_date), mdates.date2num(end_date))
        ax.set_xlabel(f"Date ({LOCAL_TZ})")
        ax.set_title(f"Station {sid} data availability (hourly)")

        ax.xaxis.set_major_locator(mdates.YearLocator(1))
        ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
        ax.xaxis.set_minor_locator(mdates.MonthLocator())
        ax.grid(True, which="major", axis="x", linestyle=":")

        ax.set_ylim(-2, y_sw + h + 2)
        fig.tight_layout()

        out_fp = OUT_DIR / f"station_{sid}_availability.png"
        fig.savefig(out_fp, dpi=200)
        plt.close(fig)

    print("Done.")


if __name__ == "__main__":
    try:
        main()
    except Exception as e:
        print(f"ERROR: {e}", file=sys.stderr)
        sys.exit(1)


Princeton stations: 181
Stormwatch stations available: 162
Total stations to plot (union): 186
Output dir: /mnt/12TB/Sujan/Spatial_correlation/Codes/Spatial_correlation
Done.
